# PunaPulse SAR · NISAR pipeline (Google Colab)

Runs the whole pipeline for the **Salar de Olaroz-Cauchari** AOI and gives you `docs/data/` to upload to the GitHub repo.

1. Get an Earthdata token: https://urs.earthdata.nasa.gov → *Generate Token*.
2. Run the cells top to bottom. GCOV files are several GB each; start with `--max-scenes 2` or `3`.


In [ ]:
# 1) Get the code: set your repo URL (or upload the zip and unzip it instead)
REPO_URL = "https://github.com/HyNuwa/punapulse-sar.git"
!git clone -q $REPO_URL punapulse-sar || true
%cd punapulse-sar
!pip install -q -r requirements.txt

In [ ]:
# 2) Earthdata login (token is kept only in this session)
import os, getpass
os.environ["EARTHDATA_TOKEN"] = getpass.getpass("Earthdata token: ")

In [ ]:
# 3) Search NISAR GCOV over the AOI (no login needed). Check dates, maturity and size before downloading.
!python -m pipeline.search_nisar --product GCOV --max-scenes 3
# Recommended one-year stack (same track, same processing, dual-pol), ~8 GB:
!python -m pipeline.search_nisar --product GCOV --maturity PROVISIONAL --track 169 --frame 100 --dates 2025-11-09,2026-02-13,2026-06-13,2026-10-11

In [ ]:
# 4) Download the selected stack (one track, one maturity level)
!python -m pipeline.download_nisar --product GCOV

In [ ]:
# 5) Process: composites, low-backscatter mask, change maps, hectares per zone
!python -m pipeline.process_gcov
# If ponds/brine are clearly visible but the mask misses them, tune the threshold, e.g.:
# !python -m pipeline.process_gcov --dark-db -18

In [ ]:
# 6) Optional, exploratory: one interferogram (GUNW) - coherence + relative phase
# !python -m pipeline.search_nisar --product GUNW --max-scenes 3
# !python -m pipeline.download_nisar --product GUNW --limit 1
# !python -m pipeline.process_gunw

In [ ]:
# 7) Rain context from NASA POWER
!python -m pipeline.fetch_precip --start 2025-01-01

In [ ]:
# 8) Quick look at the results
import json
from IPython.display import Image, display
m = json.load(open("docs/data/manifest.json"))
for s in m["scenes"]:
    print(s["date"], s["maturity"], "low-backscatter:", s["stats"]["dark_ha"], "ha")
for c in m["changes"]:
    print(c["from"], "->", c["to"], c["stats"])
print(m.get("warnings"))
if m["scenes"]:
    display(Image(filename="docs/" + m["scenes"][-1]["layers"].get("rgb", m["scenes"][-1]["layers"]["hh"]), width=400))

In [ ]:
# 9) Download docs/data as a zip, then upload its contents to docs/data/ in your GitHub repo
!cd docs && zip -qr ../punapulse_data.zip data
from google.colab import files
files.download("punapulse_data.zip")